In [ ]:
import json
from concurrent.futures import Future, ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Literal

import pandas as pd
from IPython.display import Markdown, display
from langchain_anthropic import ChatAnthropic
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from recruit import Candidates, CandidatesDB

# Set global cache to in-memory
set_llm_cache(InMemoryCache())

In [ ]:
resumes_db = Candidates()

In [ ]:
def resume_to_markdown(resume: dict[str, Any]) -> str:
    if not resume:
        return "*No resume data available.*"

    lines: list[str] = []

    # Name and Headline
    name = (
        resume.get("full_name")
        or f"{resume.get('first_name', '')} {resume.get('last_name', '')}".strip()
        or "Unknown Candidate"
    )
    lines.append(f"# {name}")

    headline = resume.get("headline")
    if headline:
        lines.append(f"**{headline}**\n")

    # Links & Location
    meta_info: list[str] = []
    linkedin_url = resume.get("linkedin_url")
    if linkedin_url:
        meta_info.append(f"[LinkedIn Profile]({linkedin_url})")

    location = resume.get("location_full") or resume.get("location_city")
    if location:
        meta_info.append(location)

    if meta_info:
        lines.append(" | ".join(meta_info) + "\n")

    # Summary
    summary = resume.get("summary")
    if summary:
        lines.append("## Summary\n")
        lines.append(f"{summary}\n")

    # Experience / Latest Positions
    experiences = resume.get("experience") or []
    if experiences:
        lines.append("## Experience\n")
        for exp in experiences:
            title = exp.get("position_title") or "Position"
            company = exp.get("company_name") or "Company"

            date_from = exp.get("date_from") or (
                str(exp.get("date_from_year")) if exp.get("date_from_year") else ""
            )
            date_to = exp.get("date_to") or (
                str(exp.get("date_to_year")) if exp.get("date_to_year") else ""
            )
            if not date_to:
                date_to = "Present" if exp.get("active_experience") else ""

            date_range = (
                f"{date_from} – {date_to}"
                if date_from and date_to
                else (date_from or date_to)
            )
            loc = exp.get("location")
            loc_str = f" | {loc}" if loc else ""

            lines.append(f"### {title} – {company}")
            if date_range or loc_str:
                lines.append(f"*{date_range}{loc_str}*\n")

            desc = exp.get("description")
            if desc:
                lines.append(f"{desc.strip()}\n")

    # Education History
    education_entries = resume.get("education") or []
    if education_entries:
        lines.append("## Education\n")
        for edu in education_entries:
            degree = edu.get("degree") or "Degree"
            institution = edu.get("institution_name") or "Institution"

            date_from_year = edu.get("date_from_year")
            date_to_year = edu.get("date_to_year")
            if date_from_year and date_to_year:
                date_str = f"{date_from_year} – {date_to_year}"
            elif date_from_year or date_to_year:
                date_str = str(date_from_year or date_to_year)
            else:
                date_str = ""

            lines.append(f"### {institution}")
            degree_line = f"**{degree}**"
            if date_str:
                degree_line += f" *({date_str})*"
            lines.append(f"{degree_line}\n")

            desc = edu.get("description")
            if desc:
                lines.append(f"{desc.strip()}\n")

    # Skills
    skills = resume.get("inferred_skills") or []
    if skills:
        lines.append("## Inferred Skills\n")
        lines.append(", ".join(skills) + "\n")

    # Languages
    languages = resume.get("languages") or []
    if languages:
        lines.append("## Languages\n")
        lang_items = []
        for lang in languages:
            lang_name = lang.get("language")
            proficiency = lang.get("proficiency")
            if lang_name and proficiency:
                lang_items.append(f"- **{lang_name}**: {proficiency}")
            elif lang_name:
                lang_items.append(f"- {lang_name}")
        if lang_items:
            lines.append("\n".join(lang_items) + "\n")

    return "\n".join(lines)

In [ ]:
resume = resumes_db.get_person(485761352) or {}
display(Markdown(resume_to_markdown(resume)))

In [ ]:
def calculate_candidate_score(assessment: dict) -> int:
    """
    Calculate the overall score for the candidate based on required and nice-to-have skills.
    Required skills are weighted more heavily than nice-to-have skills.
    """

    score = 0

    for eval in assessment["required"]:
        match eval["rating"]:
            case "strong":
                score += 3 * 2
            case "moderate":
                score += 2 * 2
            case "weak":
                score += 1 * 2

    for eval in assessment["preferred"]:
        match eval["rating"]:
            case "strong":
                score += 3
            case "moderate":
                score += 2
            case "weak":
                score += 1
            case "none":
                score -= 3
            case _:
                score += 0

    return score

In [ ]:
def has_required_criteria(candidate: dict) -> bool:
    """
    Check if the candidate meets all required criteria.
    Returns True if all required criteria are rated as 'strong' or 'moderate'.
    """

    for eval in candidate["required"]:
        if eval["rating"] not in ["strong", "moderate"]:
            return False

    return True

In [ ]:
import json

assessments_db = CandidatesDB("../spi/assessments")

assessments = []

for id, assessment in assessments_db.people.items():
    resume = resumes_db.get_person(id)
    # print(json.dumps(assessment, indent=2))
    record = {
        "id": assessment.get("id"),
        "full_name": assessment.get("full_name"),
        "linkedin_url": resume.get("linkedin_url"),
        "score": calculate_candidate_score(assessment),
        "meets_required_criteria": has_required_criteria(assessment),
    }

    for qualification in assessment["required"]:
        record |= {
            f"{qualification['name']} (Required)": qualification["rating"],
        }

    for qualification in assessment["preferred"]:
        record |= {
            f"{qualification['name']} (Preferred)": qualification["rating"],
        }

    assessments.append(record)

    # print(json.dumps(record, indent=2))
    # break

In [ ]:
print(json.dumps(assessments_db.people[489871763], indent=2))

In [ ]:
assessments_df = pd.DataFrame(assessments)
assessments_df = assessments_df.sort_values(
    by="score",
    ascending=False,
).reset_index(drop=True)
assessments_df.to_csv("../spi/short_list.csv", index=False)
assessments_df

## Draft Outreach

In [ ]:
short_list = assessments_df[assessments_df["meets_required_criteria"]].head(20)
short_list

In [ ]:
class Outreach(BaseModel):
    formal: str = Field(
        ...,
        description="A conservative outreach message",
    )
    creative: str = Field(
        ...,
        description="A creative outreach message to the candidate",
    )

    def __str__(self) -> str:
        return "\n\n---\n\n".join(
            [
                f"## Formal\n{self.formal}\n\n",
                f"## Creative\n{self.creative}\n",
            ]
        )

In [ ]:
llm = ChatOpenAI(
    model="gpt-5.6-sol",
    temperature=1.0,
)
outreach_agent = llm.with_structured_output(
    Outreach,
    method="json_schema",
)

In [ ]:
mission_path = Path("..") / "docs" / "hash_ai" / "mission.md"
with open(mission_path, "r") as file:
    mission_mk = file.read()

In [ ]:
openenings = Path("..") / "openings"
# job_description_path = openenings / "frontend-engineer-berlin.md"
job_description_path = openenings / "technical-founders-associate-berlin.md"
# job_description_path = openenings / "ai-success-engineer-berlin.md"
# job_description_path = openenings / "president-and-coo-london-berlin.md"

with open(job_description_path, "r") as file:
    job_description = file.read()

outreach_prompt = f"""
# Task
- Reach out to the candidate and encourage them to apply for the job.
- Avoid using "—" in the message.
- Create different drafts varying in tone and style, but all should be professional and respectful.
- Include links to the job application page and the company website in the message.
- Include that the candidate can send over their resume directly, skipping the first screening round.

## Styles
- Formal:
    - Use a professional and respectful tone, with clear and concise language.
- Creative:
    - Use a more engaging tone, with a touch of humor, personality storytelling, or other creative elements to make the message stand out.

---

## Company Mission and Background

{mission_mk}

---

## Basic guidelines for engaging messages

# Guidelines for Writing Engaging Messages

## Core principles

1. **Be clear immediately**

   * Make the recipient understand what your message is about within the first sentence.
   * Avoid making them decode your intent.

2. **Keep it simple**

   * Reduce the mental effort required to understand or respond.
   * Simple does not mean short; a longer message can work if every part is relevant.

3. **Be specific**

   * Say exactly what you want, why you're contacting them, and—when relevant—when you want it.
   * Replace vague statements with concrete details.

4. **Avoid artificial mystery**

   * Don't use cryptic, coy, deliberately vague, or overly aloof language to appear interesting.
   * Mystery often creates confusion rather than curiosity.

5. **Give the recipient a clear reason to care**

   * Explain the relevant context, benefit, connection, or personal reason for reaching out.

6. **Make the next step obvious**

   * End with a simple, actionable question or request.
   * Prefer questions that can be answered quickly.

7. **Respect their time**

   * If you're asking for someone's time, state the expected commitment.
   * Make the value of the interaction clear.

8. **Personalize with something concrete**

   * Reference something specific about the person, their work, or something they've said/done.
   * Generic praise is weaker than a genuine, specific observation.

9. **Don't confuse attention with complexity**

   * The goal isn't to make someone wonder *"What does this mean?"*
   * The goal is to make them think *"I understand this, and I want to respond."*

## Useful message structure

> **Context → Specific reason → Clear value/ask → Simple question**

### Example

**Weak:**

> "I have an idea I think you'll find interesting. Let's talk."

**Better:**

> "I saw your recent work on X. I have an idea for how you could use it to reach Y. Would you be open to a 15-minute call next week?"

## Make questions easy to answer

Instead of:

> "What do you think about working together?"

Try:

> "Are you open to taking on a new project this month?"

Instead of:

> "What do you want for dinner?"

Try:

> "Want tacos tonight?"

## When asking for someone's time

Include:

* **Why them**
* **What you want**
* **How long it will take**
* **What they get from it**
* **A simple yes/no or concrete-choice question**

### Example

> "I've followed your interviews and think you'd be a great fit for my podcast. We have 50,000 subscribers, and interviews run about 45 minutes. Could we book an episode this month?"

## When giving appreciation

Avoid generic praise:

> "You're awesome. Keep it up!"

Make it personal and specific:

> "I've applied your advice about X at work, and it has noticeably improved Y. I wanted to thank you."

## Quick checklist

* [ ] Is my purpose obvious?
* [ ] Is the message specific rather than vague?
* [ ] Have I removed unnecessary mystery?
* [ ] Does the recipient know why this matters?
* [ ] Is my request easy to understand?
* [ ] Is the next step obvious?
* [ ] Can they answer without much effort?
* [ ] If I'm asking for time, have I stated the commitment?
* [ ] Does the message contain something genuinely relevant to *this* person?

**Rule of thumb:** Make the message **clear, concise, specific, relevant, and easy to respond to.**
""".strip()

In [ ]:
# ## Structure
# - Start with a short greeting something like "Hi [Candidate Name],".
# - Summarize the company's mission and what the role entails in a few sentences.
# - Shortly highlight why the candidate is an exceptional good fit that would excel in the role based on their experience and skills.
# - End with a call to action, encouraging the candidate to ask questions. Express that if they are interested they can send their resume (skip first screening round).

In [ ]:
outreaches = []

candidate_prompt = """
{job_description}

---

Resume:
```json
{resume_json}
```

---

Assessment:
```json
{assessment_json}
```
""".strip()

for _, candidate in short_list.iterrows():
    # for candidate in [{"id": 485761352}]:
    resume = resumes_db.get_person(candidate["id"])
    assessment = assessments_db.get(candidate["id"])

    messages = [
        ("system", outreach_prompt),
        (
            "human",
            candidate_prompt.format(
                job_description=job_description,
                resume_json=json.dumps(resume, indent=2),
                assessment_json=json.dumps(assessment, indent=2),
            ),
        ),
    ]

    raw = outreach_agent.invoke(messages)
    outreach = Outreach.model_validate(raw)
    display(Markdown(str(outreach)))
    break